# 🥇 Template: Gold Layer - Dimensões com SCD Tipo 2

## 📋 Objetivo
Este template cria dimensões SCD Tipo 2 otimizadas para Data Warehouse seguindo melhores práticas de Business Intelligence.

## 🎯 Características das Dimensões
- **Chaves Surrogate**: IDs únicos e estáveis para performance
- **Chaves Naturais**: Referências originais do negócio
- **SCD Tipo 2**: Histórico completo de mudanças
- **Business Intelligence**: Atributos derivados para análise
- **Registros Unknown**: Tratamento de chaves órfãs

## 🔧 Configurações
Adapte as dimensões abaixo para seu modelo de dados específico.

In [ ]:
# 📦 Importações e Configurações Iniciais
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import logging
from datetime import datetime, date, timedelta
from functools import reduce

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Inicializar Spark session
spark = SparkSession.builder.appName("GoldDimensions").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("✅ Configurações iniciais completas")

In [ ]:
# 🎯 Configurações do Projeto
# TODO: Personalize estas configurações para seu projeto

# Configurações do Lakehouse
LAKEHOUSE_PATH = "/lakehouse/default/"
SILVER_PATH = f"{LAKEHOUSE_PATH}Tables/silver"
GOLD_PATH = f"{LAKEHOUSE_PATH}Tables/gold"

# Configurações SCD Tipo 2
DEFAULT_START_DATE = date(1900, 1, 1)
DEFAULT_END_DATE = date(9999, 12, 31)
PROCESSING_DATE = date.today()
UNKNOWN_ID = -1

# Dimensões para criar
# TODO: Definir dimensões específicas do seu domínio
DIMENSIONS_CONFIG = {
    "DimCustomer": {
        "source_table": "silver_customers",
        "natural_key": "customer_id",
        "business_key": "customer_id",
        "attributes": [
            "customer_name", "email", "phone", "address", 
            "customer_segment", "registration_date"
        ],
        "derived_attributes": True,
        "partition_columns": ["effective_year"]
    },
    "DimProduct": {
        "source_table": "silver_products",
        "natural_key": "product_id",
        "business_key": "product_id",
        "attributes": [
            "product_name", "category", "subcategory", "brand",
            "price", "description", "price_category"
        ],
        "derived_attributes": True,
        "partition_columns": ["effective_year"]
    },
    "DimSeller": {
        "source_table": "silver_sellers", 
        "natural_key": "seller_id",
        "business_key": "seller_id",
        "attributes": [
            "seller_name", "seller_city", "seller_state",
            "seller_zip_code", "registration_date"
        ],
        "derived_attributes": True,
        "partition_columns": ["effective_year"]
    },
    "DimGeolocation": {
        "source_table": "silver_geolocation",
        "natural_key": "zip_code",
        "business_key": "zip_code", 
        "attributes": [
            "city", "state", "latitude", "longitude",
            "region", "country"
        ],
        "derived_attributes": True,
        "partition_columns": ["effective_year"]
    }
}

print(f"📅 Data de processamento: {PROCESSING_DATE}")
print(f"🥈 Caminho Silver: {SILVER_PATH}")
print(f"🥇 Caminho Gold: {GOLD_PATH}")
print(f"📊 Dimensões a processar: {len(DIMENSIONS_CONFIG)}")

In [ ]:
# 🔧 Funções para Criação de Dimensões

def generate_dimension_surrogate_key(df, dimension_name, natural_key_col):
    """
    Gera chave surrogate única para dimensão
    
    Args:
        df: DataFrame PySpark
        dimension_name: Nome da dimensão (ex: DimCustomer)
        natural_key_col: Coluna da chave natural
    
    Returns:
        DataFrame com chave surrogate
    """
    sk_column = f"{dimension_name}SK"
    
    # Usar row_number para garantir unicidade sequencial
    window_spec = Window.orderBy(natural_key_col, "effective_date", "version")
    
    return df.withColumn(
        sk_column,
        row_number().over(window_spec)
    )

def add_business_intelligence_attributes(df, dimension_name):
    """
    Adiciona atributos de business intelligence derivados
    
    Args:
        df: DataFrame PySpark
        dimension_name: Nome da dimensão
    
    Returns:
        DataFrame com atributos BI
    """
    # TODO: Implementar atributos específicos por dimensão
    
    if dimension_name == "DimCustomer":
        df = df.withColumn("customer_age_group",
                          when(col("customer_age") < 25, "18-24")
                          .when(col("customer_age") < 35, "25-34")
                          .when(col("customer_age") < 45, "35-44")
                          .when(col("customer_age") < 55, "45-54")
                          .when(col("customer_age") < 65, "55-64")
                          .otherwise("65+")) \
               .withColumn("is_premium_customer", 
                          col("customer_segment") == "VIP") \
               .withColumn("customer_tenure_years",
                          round(datediff(current_date(), col("registration_date")) / 365.25, 1)) \
               .withColumn("email_domain",
                          split(col("email"), "@").getItem(1))
    
    elif dimension_name == "DimProduct":
        df = df.withColumn("is_high_value_product",
                          col("price") >= 500) \
               .withColumn("product_name_length",
                          length(col("product_name"))) \
               .withColumn("has_brand",
                          col("brand").isNotNull() & (col("brand") != "")) \
               .withColumn("price_range",
                          when(col("price") < 50, "$0-49")
                          .when(col("price") < 100, "$50-99")
                          .when(col("price") < 200, "$100-199")
                          .when(col("price") < 500, "$200-499")
                          .otherwise("$500+"))
    
    elif dimension_name == "DimSeller":
        df = df.withColumn("seller_region",
                          when(col("seller_state").isin(["SP", "RJ", "MG", "ES"]), "Southeast")
                          .when(col("seller_state").isin(["RS", "SC", "PR"]), "South")
                          .when(col("seller_state").isin(["GO", "MT", "MS", "DF"]), "Central-West")
                          .when(col("seller_state").isin(["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"]), "Northeast")
                          .otherwise("North")) \
               .withColumn("seller_tenure_years",
                          round(datediff(current_date(), col("registration_date")) / 365.25, 1)) \
               .withColumn("is_new_seller",
                          col("seller_tenure_years") <= 1)
    
    elif dimension_name == "DimGeolocation":
        df = df.withColumn("coordinates",
                          concat(col("latitude"), lit(","), col("longitude"))) \
               .withColumn("is_metropolitan",
                          col("city").isin(["São Paulo", "Rio de Janeiro", "Brasília", "Salvador", "Fortaleza"])) \
               .withColumn("geographic_region",
                          when(col("state").isin(["SP", "RJ", "MG", "ES"]), "Southeast")
                          .when(col("state").isin(["RS", "SC", "PR"]), "South")
                          .when(col("state").isin(["GO", "MT", "MS", "DF"]), "Central-West")
                          .when(col("state").isin(["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"]), "Northeast")
                          .otherwise("North"))
    
    return df

def create_unknown_record(dimension_name, config):
    """
    Cria registro "Unknown" para tratamento de chaves órfãs
    
    Args:
        dimension_name: Nome da dimensão
        config: Configuração da dimensão
    
    Returns:
        DataFrame com registro Unknown
    """
    sk_column = f"{dimension_name}SK"
    natural_key = config['natural_key']
    business_key = config['business_key']
    
    # Criar dados do registro Unknown
    unknown_data = {
        sk_column: UNKNOWN_ID,
        natural_key: "UNKNOWN",
        business_key: "UNKNOWN",
        "effective_date": DEFAULT_START_DATE,
        "end_date": DEFAULT_END_DATE,
        "is_current": True,
        "version": 1,
        "created_date": datetime.now(),
        "updated_date": datetime.now(),
        "effective_year": DEFAULT_START_DATE.year,
        "effective_month": DEFAULT_START_DATE.month
    }
    
    # Adicionar atributos com valores "Unknown" ou padrão
    for attr in config['attributes']:
        if "date" in attr.lower():
            unknown_data[attr] = DEFAULT_START_DATE
        elif "price" in attr.lower() or "value" in attr.lower() or "amount" in attr.lower():
            unknown_data[attr] = 0.0
        elif "id" in attr.lower():
            unknown_data[attr] = "UNKNOWN"
        else:
            unknown_data[attr] = "Unknown"
    
    # Criar DataFrame
    unknown_df = spark.createDataFrame([Row(**unknown_data)])
    
    # Adicionar atributos BI se necessário
    if config.get('derived_attributes', False):
        unknown_df = add_business_intelligence_attributes(unknown_df, dimension_name)
    
    return unknown_df

def create_dimension_scd2(dimension_name, config):
    """
    Cria dimensão completa com SCD Tipo 2
    
    Args:
        dimension_name: Nome da dimensão
        config: Configuração da dimensão
    
    Returns:
        DataFrame da dimensão
    """
    try:
        logger.info(f"🔄 Criando dimensão: {dimension_name}")
        
        # 1. Carregar dados Silver
        source_table = config['source_table']
        df_source = spark.table(source_table)
        
        # 2. Selecionar apenas registros correntes
        df_current = df_source.filter(col("is_current") == True)
        
        # 3. Selecionar colunas relevantes
        natural_key = config['natural_key']
        business_key = config['business_key']
        attributes = config['attributes']
        
        select_columns = [
            natural_key, business_key,
            "effective_date", "end_date", "is_current", "version",
            "created_date", "updated_date", "effective_year", "effective_month"
        ] + attributes
        
        df_selected = df_current.select(*[col(c) for c in select_columns if c in df_current.columns])
        
        # 4. Adicionar atributos de business intelligence
        if config.get('derived_attributes', False):
            df_enhanced = add_business_intelligence_attributes(df_selected, dimension_name)
        else:
            df_enhanced = df_selected
        
        # 5. Gerar chave surrogate
        df_with_sk = generate_dimension_surrogate_key(df_enhanced, dimension_name, natural_key)
        
        # 6. Criar registro Unknown
        df_unknown = create_unknown_record(dimension_name, config)
        
        # 7. Combinar dados regulares com Unknown
        # Ajustar SK para começar após o Unknown
        max_sk = df_with_sk.agg(max(f"{dimension_name}SK")).collect()[0][0] or 0
        df_with_sk_adjusted = df_with_sk.withColumn(
            f"{dimension_name}SK",
            col(f"{dimension_name}SK") + max_sk + 1
        )
        
        # Union dos dados
        df_final = df_unknown.unionByName(df_with_sk_adjusted, allowMissingColumns=True)
        
        # 8. Adicionar metadados da dimensão
        df_final = df_final.withColumn("dimension_name", lit(dimension_name)) \
                          .withColumn("source_system", lit(source_table)) \
                          .withColumn("processing_date", lit(PROCESSING_DATE))
        
        record_count = df_final.count()
        logger.info(f"✅ {dimension_name} criada: {record_count:,} registros")
        
        return df_final
        
    except Exception as e:
        logger.error(f"❌ Erro ao criar {dimension_name}: {str(e)}")
        raise

def create_dim_date(start_date=None, end_date=None):
    """
    Cria dimensão temporal (DimDate) completa
    
    Args:
        start_date: Data inicial (padrão: 2000-01-01)
        end_date: Data final (padrão: 2030-12-31)
    
    Returns:
        DataFrame da dimensão temporal
    """
    if start_date is None:
        start_date = date(2000, 1, 1)
    if end_date is None:
        end_date = date(2030, 12, 31)
    
    logger.info(f"🔄 Criando DimDate: {start_date} até {end_date}")
    
    # Gerar sequência de datas
    date_range = []
    current_date = start_date
    
    while current_date <= end_date:
        date_range.append(current_date)
        current_date += timedelta(days=1)
    
    # Criar DataFrame
    df_dates = spark.createDataFrame(
        [(d,) for d in date_range],
        ["date_value"]
    )
    
    # Adicionar atributos temporais
    df_dim_date = df_dates.withColumn("DimDateSK", 
                                     date_format(col("date_value"), "yyyyMMdd").cast("int")) \
        .withColumn("year", year(col("date_value"))) \
        .withColumn("month", month(col("date_value"))) \
        .withColumn("day", dayofmonth(col("date_value"))) \
        .withColumn("quarter", quarter(col("date_value"))) \
        .withColumn("week_of_year", weekofyear(col("date_value"))) \
        .withColumn("day_of_week", dayofweek(col("date_value"))) \
        .withColumn("day_of_year", dayofyear(col("date_value"))) \
        .withColumn("month_name", date_format(col("date_value"), "MMMM")) \
        .withColumn("month_name_short", date_format(col("date_value"), "MMM")) \
        .withColumn("day_name", date_format(col("date_value"), "EEEE")) \
        .withColumn("day_name_short", date_format(col("date_value"), "EEE")) \
        .withColumn("is_weekend", dayofweek(col("date_value")).isin([1, 7])) \
        .withColumn("is_month_start", dayofmonth(col("date_value")) == 1) \
        .withColumn("is_month_end", 
                   dayofmonth(col("date_value")) == dayofmonth(last_day(col("date_value")))) \
        .withColumn("is_quarter_start", 
                   (month(col("date_value")).isin([1, 4, 7, 10])) & 
                   (dayofmonth(col("date_value")) == 1)) \
        .withColumn("is_year_start", 
                   (month(col("date_value")) == 1) & (dayofmonth(col("date_value")) == 1)) \
        .withColumn("fiscal_year", 
                   when(month(col("date_value")) >= 4, year(col("date_value")))
                   .otherwise(year(col("date_value")) - 1)) \
        .withColumn("quarter_name", concat(lit("Q"), col("quarter"), lit("-"), col("year"))) \
        .withColumn("year_month", date_format(col("date_value"), "yyyy-MM")) \
        .withColumn("year_quarter", concat(col("year"), lit("-Q"), col("quarter")))
    
    # Adicionar metadados
    df_dim_date = df_dim_date.withColumn("dimension_name", lit("DimDate")) \
                            .withColumn("source_system", lit("SYSTEM_GENERATED")) \
                            .withColumn("processing_date", lit(PROCESSING_DATE))
    
    record_count = df_dim_date.count()
    logger.info(f"✅ DimDate criada: {record_count:,} registros")
    
    return df_dim_date

print("🔧 Funções para criação de dimensões definidas")

In [ ]:
# 🥇 Criação das Dimensões Gold

dimensions_summary = []

# Processar cada dimensão configurada
for dimension_name, config in DIMENSIONS_CONFIG.items():
    try:
        # Criar dimensão
        df_dimension = create_dimension_scd2(dimension_name, config)
        
        # Definir caminho da tabela Gold
        gold_table_path = f"{GOLD_PATH}/{dimension_name}"
        
        # Salvar dimensão
        df_dimension.write \
            .mode("overwrite") \
            .partitionBy(*config.get('partition_columns', [])) \
            .option("mergeSchema", "true") \
            .format("delta") \
            .save(gold_table_path)
        
        # Registrar tabela no metastore
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {dimension_name}
            USING DELTA
            LOCATION '{gold_table_path}'
        """)
        
        # Otimizar tabela
        spark.sql(f"OPTIMIZE {dimension_name}")
        
        # Coletar métricas
        record_count = df_dimension.count()
        current_records = df_dimension.filter(col("is_current") == True).count()
        unknown_records = df_dimension.filter(col(f"{dimension_name}SK") == UNKNOWN_ID).count()
        
        summary = {
            "dimension_name": dimension_name,
            "total_records": record_count,
            "current_records": current_records,
            "unknown_records": unknown_records,
            "historical_records": record_count - current_records,
            "source_table": config['source_table'],
            "processing_date": PROCESSING_DATE
        }
        dimensions_summary.append(summary)
        
        logger.info(f"✅ {dimension_name} salva: {record_count:,} total, {current_records:,} correntes")
        
        # Mostrar amostra da dimensão
        print(f"\n📋 Amostra da dimensão {dimension_name}:")
        df_dimension.filter(col("is_current") == True).limit(5).display()
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {dimension_name}: {str(e)}")
        continue

print("\n🎉 Criação das dimensões concluída!")

In [ ]:
# 📅 Criação da Dimensão Temporal (DimDate)

try:
    # Criar DimDate
    df_dim_date = create_dim_date(
        start_date=date(2000, 1, 1),
        end_date=date(2030, 12, 31)
    )
    
    # Salvar DimDate
    gold_date_path = f"{GOLD_PATH}/DimDate"
    
    df_dim_date.write \
        .mode("overwrite") \
        .partitionBy("year") \
        .option("mergeSchema", "true") \
        .format("delta") \
        .save(gold_date_path)
    
    # Registrar no metastore
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS DimDate
        USING DELTA
        LOCATION '{gold_date_path}'
    """)
    
    # Otimizar
    spark.sql("OPTIMIZE DimDate ZORDER BY (date_value)")
    
    # Adicionar às métricas
    date_record_count = df_dim_date.count()
    dimensions_summary.append({
        "dimension_name": "DimDate",
        "total_records": date_record_count,
        "current_records": date_record_count,
        "unknown_records": 0,
        "historical_records": 0,
        "source_table": "SYSTEM_GENERATED",
        "processing_date": PROCESSING_DATE
    })
    
    logger.info(f"✅ DimDate salva: {date_record_count:,} registros")
    
    # Mostrar amostra
    print(f"\n📋 Amostra da DimDate:")
    df_dim_date.filter(year(col("date_value")) == 2024).limit(5).display()
    
except Exception as e:
    logger.error(f"❌ Erro ao criar DimDate: {str(e)}")

print("\n📅 DimDate criada com sucesso!")

In [ ]:
# 📊 Relatório de Dimensões Gold

print("\n📊 RELATÓRIO DE DIMENSÕES - GOLD LAYER")
print("=" * 60)

# Converter para DataFrame para melhor visualização
summary_df = spark.createDataFrame(
    [Row(**summary) for summary in dimensions_summary]
)

# Mostrar resumo das dimensões
summary_df.select(
    "dimension_name",
    "total_records",
    "current_records",
    "historical_records",
    "unknown_records",
    "source_table",
    "processing_date"
).display()

# Calcular estatísticas gerais
total_records = sum([s['total_records'] for s in dimensions_summary])
total_current = sum([s['current_records'] for s in dimensions_summary])
total_historical = sum([s['historical_records'] for s in dimensions_summary])
total_unknown = sum([s['unknown_records'] for s in dimensions_summary])

print(f"\n📋 RESUMO GERAL:")
print(f"   📊 Total de registros em dimensões: {total_records:,}")
print(f"   🎯 Total de registros correntes: {total_current:,}")
print(f"   📚 Total de registros históricos: {total_historical:,}")
print(f"   ❓ Total de registros Unknown: {total_unknown:,}")
print(f"   📅 Data de processamento: {PROCESSING_DATE}")

# Mostrar dimensões por tamanho
largest_dimensions = sorted(dimensions_summary, key=lambda x: x['total_records'], reverse=True)
print(f"\n📈 Dimensões por tamanho:")
for dim in largest_dimensions:
    print(f"   - {dim['dimension_name']}: {dim['total_records']:,} registros")

# Identificar dimensões com muito histórico
high_history = [d for d in dimensions_summary if d['historical_records'] > d['current_records'] * 0.2]
if high_history:
    print(f"\n📈 Dimensões com alto histórico (>20% de registros históricos):")
    for dim in high_history:
        history_pct = (dim['historical_records'] / dim['total_records']) * 100
        print(f"   - {dim['dimension_name']}: {history_pct:.1f}% histórico")
else:
    print(f"\n✅ Histórico das dimensões está equilibrado")

In [ ]:
# 🔍 Validação das Dimensões

print("\n🔍 VALIDAÇÃO DAS DIMENSÕES GOLD")
print("=" * 50)

# Listar todas as tabelas Gold criadas
gold_tables = spark.sql("""
    SHOW TABLES LIKE 'Dim*'
""").collect()

print(f"📋 Dimensões Gold criadas: {len(gold_tables)}")

for table in gold_tables:
    table_name = table.tableName
    
    try:
        print(f"\n📊 Validando {table_name}:")
        
        # 1. Contar registros
        total_count = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}").collect()[0].count
        
        # 2. Verificar registros Unknown
        unknown_count = spark.sql(f"""
            SELECT COUNT(*) as count 
            FROM {table_name} 
            WHERE {table_name}SK = {UNKNOWN_ID}
        """).collect()[0].count
        
        # 3. Verificar unicidade da chave surrogate
        duplicate_sk = spark.sql(f"""
            SELECT {table_name}SK, COUNT(*) as count
            FROM {table_name}
            GROUP BY {table_name}SK
            HAVING COUNT(*) > 1
        """).count()
        
        print(f"   📊 Total de registros: {total_count:,}")
        print(f"   ❓ Registros Unknown: {unknown_count}")
        
        if duplicate_sk == 0:
            print(f"   ✅ Chaves surrogate únicas")
        else:
            print(f"   ❌ {duplicate_sk} chaves surrogate duplicadas")
        
        # 4. Para dimensões SCD2, verificar registros correntes
        if table_name != "DimDate":
            current_count = spark.sql(f"""
                SELECT COUNT(*) as count 
                FROM {table_name} 
                WHERE is_current = true
            """).collect()[0].count
            
            print(f"   🎯 Registros correntes: {current_count:,}")
            
            # Verificar se há chaves naturais com múltiplos registros correntes
            natural_key_col = None
            columns = spark.sql(f"DESCRIBE {table_name}").collect()
            for col_info in columns:
                if "_id" in col_info.col_name and "SK" not in col_info.col_name:
                    natural_key_col = col_info.col_name
                    break
            
            if natural_key_col:
                duplicate_current = spark.sql(f"""
                    SELECT {natural_key_col}, COUNT(*) as count
                    FROM {table_name}
                    WHERE is_current = true
                    GROUP BY {natural_key_col}
                    HAVING COUNT(*) > 1
                """).count()
                
                if duplicate_current == 0:
                    print(f"   ✅ Chaves naturais únicas nos registros correntes")
                else:
                    print(f"   ❌ {duplicate_current} chaves naturais com múltiplos registros correntes")
        
        # 5. Verificar partições
        try:
            partitions = spark.sql(f"SHOW PARTITIONS {table_name}").count()
            print(f"   📂 Partições: {partitions}")
        except:
            print(f"   📂 Sem particionamento")
            
    except Exception as e:
        print(f"   ❌ Erro na validação: {str(e)}")

print("\n🎉 Validação das dimensões concluída!")
print("\n➡️  Próximo passo: Criar tabelas de fatos (FactTables)")

## 📚 Próximos Passos

1. **Fatos**: Criar tabelas de fatos usando as dimensões criadas
2. **Performance**: Implementar Z-ordering e compactação
3. **Validação**: Criar testes de integridade referencial
4. **Monitoramento**: Implementar alertas para problemas de qualidade

## 🔧 Customizações Avançadas

### Para Mini-Dimensões:
```python
def create_mini_dimension(df, dimension_name, fast_changing_attrs):
    """Cria mini-dimensão para atributos que mudam frequentemente"""
    return df.select(
        ["sk_column"] + fast_changing_attrs
    ).dropDuplicates()
```

### Para Dimensões Junk:
```python
def create_junk_dimension(flags_and_indicators):
    """Cria dimensão junk para flags e indicadores"""
    # Combinar todas as possibilidades de flags
    combinations = itertools.product(*[flag_values for flag_values in flags_and_indicators.values()])
    return spark.createDataFrame(combinations, flags_and_indicators.keys())
```

### Para Dimensões Degeneradas:
```python
def add_degenerate_dimension(fact_df, degenerate_columns):
    """Adiciona dimensões degeneradas à tabela de fatos"""
    for col in degenerate_columns:
        fact_df = fact_df.withColumn(f"DD_{col}", col(col))
    return fact_df
```

### Para Bridge Tables:
```python
def create_bridge_table(many_to_many_df, group_key, member_key):
    """Cria tabela bridge para relacionamentos M:N"""
    return many_to_many_df.select(
        group_key, member_key,
        "weight_factor", "allocation_percentage"
    ).distinct()
```